##### Parameter Cell

In [1]:
batch_id = ""

StatementMeta(, 2d9acb6b-c062-4920-b6ea-be3923fc2951, 3, Finished, Available, Finished, False)

##### Setup

In [2]:
import sys
sys.path.insert(0, "/lakehouse/default/Files/code")

from bikeshare import control
from bikeshare.audit import RunLogger
from bikeshare.config import make_config

spark.conf.set("spark.sql.ansi.enabled", "false")       # bad casts become NULL (then fail a rule) instead of crashing
spark.conf.set("spark.sql.session.timeZone", "UTC")
spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "true")
cfg = make_config(batch_id=batch_id or None)
control.ensure_control_plane(spark)
log = RunLogger(spark, cfg)
print("batch:", cfg.batch_id)

StatementMeta(, 2d9acb6b-c062-4920-b6ea-be3923fc2951, 4, Finished, Available, Finished, False)

batch: b003


##### Run

In [3]:
from bikeshare import silver
silver.run(spark, cfg, log)

StatementMeta(, 2d9acb6b-c062-4920-b6ea-be3923fc2951, 5, Finished, Available, Finished, False)

  [silver ] station_versions             SUCCESS  read=2       written=2       rejected=0      41.6s
  [silver ] trips                        SUCCESS  read=15363   written=15221   rejected=71     43.6s
  [silver ] weather_hourly               SUCCESS  read=408     written=384     rejected=0      33.6s
  [silver ] events                       SUCCESS  read=0       written=0       rejected=0       5.6s


##### Checks

In [4]:
for t in ["silver.trips", "silver.quarantine_trips", "silver.station_versions", "silver.weather_hourly", "silver.events"]:
    print(t, spark.table(t).count())

# which rules rejected rows?
spark.sql("""SELECT rule, COUNT(*) AS rows_quarantined
             FROM (SELECT explode(dq_errors) AS rule FROM silver.quarantine_trips)
             GROUP BY rule ORDER BY 2 DESC""").show(truncate=False)

# every rule result, logged
spark.sql("""SELECT table_name, rule_name, severity, rows_failed, status
             FROM audit.dq_results WHERE layer = 'silver' ORDER BY table_name, rule_name""").show(60, truncate=False)

spark.table("silver.trips").select("trip_id", "started_at", "duration_sec", "dq_warnings", "_row_hash", "_batch_id", "_processed_at").show(5, truncate=False)

StatementMeta(, 2d9acb6b-c062-4920-b6ea-be3923fc2951, 6, Finished, Available, Finished, False)

silver.trips 18851
silver.quarantine_trips 92
silver.station_versions 42
silver.weather_hourly 504
silver.events 20
+--------------------+----------------+
|rule                |rows_quarantined|
+--------------------+----------------+
|end_not_before_start|31              |
|trip_id_not_null    |24              |
|rider_type_valid    |20              |
|duration_le_24h     |8               |
|fare_non_negative   |6               |
|not_future_dated    |3               |
+--------------------+----------------+

+-----------------------+-----------------------+--------+-----------+------+
|table_name             |rule_name              |severity|rows_failed|status|
+-----------------------+-----------------------+--------+-----------+------+
|silver.events          |attendance_non_negative|error   |0          |PASS  |
|silver.events          |dates_valid            |error   |0          |PASS  |
|silver.events          |event_name_not_null    |error   |0          |PASS  |
|silver.events 